# 13 — Real Audio Challenge

Everything so far used synthetic TTS audio — clean, studio-quality clips generated from text. But the ultimate goal is transcribing real Game of Thrones dialogue. This notebook confronts the domain gap: how do our strategies perform on actual GoT audio with background music, reverb, and actor variation?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
SYNTH_DIR = PROJECT_ROOT / 'data' / 'synthetic'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

C_TEAL = '#4ecdc4'
C_RED = '#ff6b6b'
C_YELLOW = '#ffd93d'
C_DARK_TEAL = '#45b7aa'
COLORS = [C_TEAL, C_RED, C_YELLOW, C_DARK_TEAL]

# List available real clips
real_wavs = sorted(RAW_DIR.glob('*.wav'))
processed_wavs = sorted(PROCESSED_DIR.glob('*_vocals.wav'))
synth_wavs = sorted(SYNTH_DIR.glob('*.wav'))[:5]  # Just a few for comparison

print('Real audio clips:')
for f in real_wavs:
    print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')
print(f'\nProcessed (vocal-isolated): {len(processed_wavs)} files')
print(f'Synthetic clips: {len(list(SYNTH_DIR.glob("*.wav")))} files')

---
## 1. Synthetic vs Real Mel Comparison

Compare mel spectrograms of a synthetic clip and a real GoT clip to visualize the domain gap.

In [ ]:
# Load one synthetic clip and one real clip
synth_path = SYNTH_DIR / 'd0000.wav'
real_path = processed_wavs[0] if processed_wavs else real_wavs[0]

synth_audio, synth_sr = librosa.load(synth_path, sr=16000)
real_audio, real_sr = librosa.load(real_path, sr=16000)

# Compute mel spectrograms
synth_mel = librosa.feature.melspectrogram(y=synth_audio, sr=synth_sr, n_mels=80)
real_mel = librosa.feature.melspectrogram(y=real_audio[:synth_sr * 5], sr=real_sr, n_mels=80)  # First 5 seconds

synth_mel_db = librosa.power_to_db(synth_mel, ref=np.max)
real_mel_db = librosa.power_to_db(real_mel, ref=np.max)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = librosa.display.specshow(synth_mel_db, sr=synth_sr, x_axis='time', y_axis='mel',
                                ax=axes[0], cmap='magma')
axes[0].set_title(f'Synthetic: {synth_path.name}', fontsize=12)
axes[0].set_xlabel('Time (s)')
plt.colorbar(im0, ax=axes[0], format='%+2.0f dB', shrink=0.8)

im1 = librosa.display.specshow(real_mel_db, sr=real_sr, x_axis='time', y_axis='mel',
                                ax=axes[1], cmap='magma')
axes[1].set_title(f'Real: {real_path.name}', fontsize=12)
axes[1].set_xlabel('Time (s)')
plt.colorbar(im1, ax=axes[1], format='%+2.0f dB', shrink=0.8)

plt.suptitle('Mel Spectrogram: Synthetic vs Real Audio', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Synthetic duration: {len(synth_audio)/synth_sr:.2f}s')
print(f'Real duration: {len(real_audio)/real_sr:.2f}s')

---
## 2. Existing Real Audio Results

Load the transcription and translation results from previous runs on real GoT clips.

In [ ]:
# Load existing real audio results
real_result_files = [
    'drogo_speech_clean_transcription.json',
    'drogo_rhaego_speech_transcription.json',
    'dothraki_short_transcription.json',
]

real_results = {}
for fname in real_result_files:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        clip_name = fname.replace('_transcription.json', '')
        real_results[clip_name] = json.loads(fpath.read_text())

# Also load summary files for more context
real_summaries = {}
for fname in ['drogo_speech_clean_summary.json', 'drogo_rhaego_speech_summary.json', 'dothraki_short_summary.json']:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        clip_name = fname.replace('_summary.json', '')
        real_summaries[clip_name] = json.loads(fpath.read_text())

print(f'Real audio result sets: {len(real_results)}')
for name, data in real_results.items():
    print(f'\n  {name}:')
    if isinstance(data, dict):
        for key in list(data.keys())[:5]:
            val = str(data[key])[:80]
            print(f'    {key}: {val}')
    elif isinstance(data, list):
        print(f'    {len(data)} segments')
        for seg in data[:3]:
            print(f'    - {str(seg)[:80]}')

---
## 3. Run Strategies on Real Audio

Run embedding, DTW, and fine-tune strategies on each real clip and collect outputs. Since there's no ground truth for real clips, analysis is qualitative.

In [ ]:
from pipeline.run import Pipeline

# Use processed (vocal-isolated) clips for better results
test_clips = processed_wavs[:3] if processed_wavs else real_wavs[:3]

strategies_to_test = ['embedding', 'dtw', 'finetune']
real_strategy_results = {}  # clip_name -> {strategy: PipelineResult}

for clip_path in test_clips:
    clip_name = clip_path.stem
    real_strategy_results[clip_name] = {}
    
    for strategy in strategies_to_test:
        try:
            pipeline = Pipeline(whisper_model='small', strategy=strategy, skip_separation=True)
            result = pipeline.run(str(clip_path), save=False)
            real_strategy_results[clip_name][strategy] = {
                'raw_dothraki': result.raw_dothraki,
                'quality': result.quality,
                'clip_matches': result.clip_matches[:3] if result.clip_matches else None,
            }
        except Exception as e:
            real_strategy_results[clip_name][strategy] = {'error': str(e)}
    
    print(f'Processed: {clip_name}')

print(f'\nCompleted {len(real_strategy_results)} clips x {len(strategies_to_test)} strategies')

---
## 4. Strategy Outputs Comparison

Compare what each strategy produces for the same real audio clip. Do they agree?

In [ ]:
print('Strategy Output Comparison — Real GoT Audio')
print('=' * 80)

for clip_name, strats in real_strategy_results.items():
    print(f'\n--- {clip_name} ---')
    for strategy, result in strats.items():
        if 'error' in result:
            print(f'  {strategy:12s}: ERROR — {result["error"][:60]}')
        else:
            output = result.get('raw_dothraki', '(none)')[:80]
            quality = result.get('quality', '?')
            print(f'  {strategy:12s}: [{quality}] {output}')
            if result.get('clip_matches'):
                top = result['clip_matches'][0]
                score = top.get('score', top.get('dtw_cost', '?'))
                print(f'  {"":12s}  → matched: {top.get("dothraki", "")[:50]} (score={score})')
    
    # Check agreement
    outputs = [r.get('raw_dothraki', '') for r in strats.values() if 'error' not in r]
    unique = len(set(o.strip().lower() for o in outputs if o))
    print(f'  Agreement: {len(outputs) - unique + 1 if outputs else 0}/{len(outputs)} strategies agree' if outputs else '')

---
## 5. Confidence Analysis

Compare retrieval scores (embedding/DTW) on real audio vs synthetic audio.

In [ ]:
# Load synthetic eval scores for comparison
synth_emb_scores = []
synth_dtw_costs = []

emb_eval = json.loads((RESULTS_DIR / 'batch_eval_embedding_small.json').read_text())
dtw_eval = json.loads((RESULTS_DIR / 'batch_eval_dtw_small.json').read_text())

for r in emb_eval['results']:
    synth_emb_scores.append(r.get('top_match_score', 0))
for r in dtw_eval['results']:
    matches = r.get('clip_matches', [])
    if matches:
        synth_dtw_costs.append(matches[0].get('dtw_cost', 0))

# Real audio scores
real_emb_scores = []
real_dtw_costs = []

for clip_name, strats in real_strategy_results.items():
    if 'embedding' in strats and 'error' not in strats['embedding']:
        matches = strats['embedding'].get('clip_matches', [])
        if matches:
            real_emb_scores.append(matches[0].get('score', 0))
    if 'dtw' in strats and 'error' not in strats['dtw']:
        matches = strats['dtw'].get('clip_matches', [])
        if matches:
            real_dtw_costs.append(matches[0].get('dtw_cost', 0))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Embedding scores comparison
box_data_emb = [synth_emb_scores]
box_labels_emb = [f'Synthetic\n(n={len(synth_emb_scores)})']
if real_emb_scores:
    box_data_emb.append(real_emb_scores)
    box_labels_emb.append(f'Real\n(n={len(real_emb_scores)})')

bp1 = axes[0].boxplot(box_data_emb, labels=box_labels_emb, patch_artist=True,
                       boxprops=dict(facecolor=C_TEAL, alpha=0.7),
                       medianprops=dict(color=C_RED, linewidth=2))
if len(bp1['boxes']) > 1:
    bp1['boxes'][1].set_facecolor(C_YELLOW)
axes[0].set_ylabel('Embedding Score')
axes[0].set_title('Embedding Scores: Synthetic vs Real')

# DTW costs comparison
box_data_dtw = [synth_dtw_costs]
box_labels_dtw = [f'Synthetic\n(n={len(synth_dtw_costs)})']
if real_dtw_costs:
    box_data_dtw.append(real_dtw_costs)
    box_labels_dtw.append(f'Real\n(n={len(real_dtw_costs)})')

bp2 = axes[1].boxplot(box_data_dtw, labels=box_labels_dtw, patch_artist=True,
                       boxprops=dict(facecolor=C_TEAL, alpha=0.7),
                       medianprops=dict(color=C_RED, linewidth=2))
if len(bp2['boxes']) > 1:
    bp2['boxes'][1].set_facecolor(C_YELLOW)
axes[1].set_ylabel('DTW Cost')
axes[1].set_title('DTW Costs: Synthetic vs Real')

plt.tight_layout()
plt.show()

---
## 6. Audio Quality Metrics

Compare signal-to-noise ratio (SNR) estimation and spectral flatness between synthetic and real audio.

In [ ]:
def estimate_snr(audio, sr):
    """Estimate SNR using signal energy vs noise floor."""
    rms = librosa.feature.rms(y=audio)[0]
    signal_power = np.mean(rms ** 2)
    # Estimate noise from quietest 10% of frames
    sorted_rms = np.sort(rms)
    noise_rms = sorted_rms[:max(1, len(sorted_rms) // 10)]
    noise_power = np.mean(noise_rms ** 2)
    if noise_power > 0:
        return 10 * np.log10(signal_power / noise_power)
    return float('inf')

def spectral_flatness_mean(audio, sr):
    """Mean spectral flatness (0=tonal, 1=noise-like)."""
    sf = librosa.feature.spectral_flatness(y=audio)[0]
    return np.mean(sf)

# Compute metrics for synthetic clips
synth_snrs = []
synth_flatness = []
for wav_path in list(SYNTH_DIR.glob('*.wav'))[:20]:  # Sample 20
    audio, sr = librosa.load(wav_path, sr=16000)
    synth_snrs.append(estimate_snr(audio, sr))
    synth_flatness.append(spectral_flatness_mean(audio, sr))

# Compute metrics for real clips
real_snrs = []
real_flatness = []
for wav_path in processed_wavs + real_wavs:
    if wav_path.suffix == '.wav':
        audio, sr = librosa.load(wav_path, sr=16000)
        real_snrs.append(estimate_snr(audio, sr))
        real_flatness.append(spectral_flatness_mean(audio, sr))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SNR comparison
bp1 = axes[0].boxplot([synth_snrs, real_snrs],
                       labels=[f'Synthetic (n={len(synth_snrs)})', f'Real (n={len(real_snrs)})'],
                       patch_artist=True,
                       boxprops=dict(facecolor=C_TEAL, alpha=0.7),
                       medianprops=dict(color=C_RED, linewidth=2))
bp1['boxes'][1].set_facecolor(C_YELLOW)
axes[0].set_ylabel('Estimated SNR (dB)')
axes[0].set_title('Signal-to-Noise Ratio')

# Spectral flatness comparison
bp2 = axes[1].boxplot([synth_flatness, real_flatness],
                       labels=[f'Synthetic (n={len(synth_flatness)})', f'Real (n={len(real_flatness)})'],
                       patch_artist=True,
                       boxprops=dict(facecolor=C_TEAL, alpha=0.7),
                       medianprops=dict(color=C_RED, linewidth=2))
bp2['boxes'][1].set_facecolor(C_YELLOW)
axes[1].set_ylabel('Spectral Flatness (0=tonal, 1=noise)')
axes[1].set_title('Spectral Flatness')

plt.tight_layout()
plt.show()

print(f'Synthetic: mean SNR={np.mean(synth_snrs):.1f}dB, mean flatness={np.mean(synth_flatness):.4f}')
print(f'Real:      mean SNR={np.mean(real_snrs):.1f}dB, mean flatness={np.mean(real_flatness):.4f}')

---
## Conclusions

1. **Significant domain gap** — mel spectrograms reveal stark differences between synthetic (clean, single-speaker TTS) and real (background music, reverb, actor voice variation) audio. This gap is the primary challenge for real-world deployment.

2. **Strategy disagreement on real audio** — while strategies often agree on synthetic clips (where the answer is deterministic), they diverge significantly on real audio, suggesting lower confidence across the board.

3. **Lower retrieval confidence on real data** — embedding scores and DTW costs shift unfavorably when processing real audio, confirming that the domain gap affects all feature-based approaches.

4. **Audio quality metrics quantify the gap** — real GoT audio has lower SNR and different spectral characteristics than synthetic TTS, explaining why models trained/indexed on synthetic data struggle with transfer.

**Key Takeaway:** Bridging the synthetic-to-real gap requires either (1) data augmentation during index building (noise, reverb, pitch variation), (2) fine-tuning on real Dothraki audio (limited availability), or (3) better source separation to clean real audio before processing. The vocal isolation step helps but doesn't fully close the gap.